### ERA5 1° monthly precipitation

A data-preparation step for the E1–E5 notebooks: builds the ERA5 1° monthly precipitation reference they read, `ERA5/mon_1deg/native6_ERA5_an_v1_Amon_pr_1978-2024.nc`. The other `mon_1deg` variables come from ESMValTool and have no `pr`.

- **Source:** ERA5 monthly-mean total precipitation at 0.25° (`ERA5/mon/ERA5_total_precipitation_mon_full_sfc_1940-2024.nc`). Monthly means of accumulated fields are in m per day.
- **Method:** conservative regrid (xESMF) onto the grid of the ESMValTool `tas` file, limited to latitudes ±87.5° to avoid regridding artifacts at the poles. Rows poleward of that are NaN. Converted to kg m⁻² s⁻¹.
- **Period:** 1978-01 to 2024-12, matching the other `mon_1deg` files. The first version of this file started in 1979-10, which left ERA5 `pr` missing for Jan 1978 – Sep 1979.

Paths are relative to `AIMIP_DATA_ROOT`, as in the evaluation notebooks. The default, `../../../local_data`, is the repo's `local_data/` as seen from this directory. Running the notebook replaces the existing output file with a new one; hard links to the old file keep the old contents.

In [1]:
import os

import numpy as np
import xarray as xr
import xesmf
from dask.diagnostics import ProgressBar

In [2]:
DATA_ROOT = os.environ.get('AIMIP_DATA_ROOT', '../../../local_data')
INPUT_PATH = os.path.join(DATA_ROOT, 'ERA5/mon/ERA5_total_precipitation_mon_full_sfc_1940-2024.nc')
TEMPLATE_PATH = os.path.join(DATA_ROOT, 'ERA5/mon_1deg/native6_ERA5_an_v1_Amon_tas_1978-2024.nc')
OUTPUT_PATH = os.path.join(DATA_ROOT, 'ERA5/mon_1deg/native6_ERA5_an_v1_Amon_pr_1978-2024.nc')
TIME_BOUNDS = ('1978-01-01', '2024-12-31')
LATITUDE_LIMITS = (-87.5, 87.5) # avoid regridding artifacts at the poles
KG_PER_M2_PER_M = 1000
SECONDS_PER_DAY = 86_400

In [3]:
precip_ds = xr.open_dataset(INPUT_PATH, chunks={})
template_ds = xr.open_dataset(TEMPLATE_PATH, chunks={})

In [4]:
destination_grid = (
    template_ds
    .sel(lat=slice(*LATITUDE_LIMITS))
    [['lat', 'lon', 'lat_bnds', 'lon_bnds']]
)
regridder = xesmf.Regridder(
    precip_ds[['lat', 'lon']],
    destination_grid,
    method='conservative'
)
src_dataarray = precip_ds.TP.sel(time=slice(*TIME_BOUNDS))
precip_regridded = regridder.regrid_dataarray(src_dataarray)

/home/brianhenn/miniconda3/envs/aimip-evaluations/lib/python3.11/site-packages/xesmf/backend.py:57: UserWarning: Latitude is outside of [-90, 90]
  warnings.warn('Latitude is outside of [-90, 90]')


In [5]:
output_ds = template_ds.sel(time=slice(*TIME_BOUNDS)).drop_vars('tas')

# source times are month starts, template times are mid-month: check the months line up before relabelling
source_months = precip_regridded.time.dt.strftime('%Y-%m').values
output_months = output_ds.time.dt.strftime('%Y-%m').values
assert np.array_equal(source_months, output_months), (source_months[[0, -1]], output_months[[0, -1]])

output_ds['pr'] = xr.full_like(template_ds['tas'].sel(time=slice(*TIME_BOUNDS)), fill_value=np.nan)
output_ds['pr'].loc[{'lat': slice(*LATITUDE_LIMITS)}] = precip_regridded.assign_coords({'time': output_ds.time})
output_ds['pr'] = output_ds['pr'] * KG_PER_M2_PER_M / SECONDS_PER_DAY # m per day to kg m-2 s-1
output_ds['pr'] = output_ds['pr'].assign_attrs({"long_name": "Surface precipitation rate", "units": "kg/m**2/s"})

# don't inherit the template's ESMValTool global attributes, which would misstate this file's provenance
output_ds.attrs = {
    'Conventions': 'CF-1.7',
    'title': 'ERA5 monthly-mean precipitation rate, 1 degree',
    'source': 'ERA5 monthly-mean total precipitation at 0.25 degree (ERA5/mon/ERA5_total_precipitation_mon_full_sfc_1940-2024.nc)',
    'history': (
        f'Conservatively regridded with xESMF {xesmf.__version__} onto the grid of '
        'native6_ERA5_an_v1_Amon_tas_1978-2024.nc, latitudes limited to +/-87.5 (NaN poleward), '
        'converted from m per day to kg m-2 s-1. Created by evaluations/notebooks/data_prep/ERA5-1deg-precipitation.ipynb.'
    ),
}

In [6]:
# every month should have data inside the latitude limits
in_domain = output_ds['pr'].sel(lat=slice(*LATITUDE_LIMITS))
all_nan_months = in_domain.isnull().all(dim=['lat', 'lon']).compute()
assert not all_nan_months.any(), output_ds.time.values[all_nan_months.values]
print(output_ds.time.values[[0, -1]], output_ds.sizes['time'], 'months')

['1978-01-17T00:00:00.000000000' '2024-12-17T00:00:00.000000000'] 564 months


In [7]:
# write to a new file and rename it into place: writing directly would truncate the existing file's inode,
# changing every hard link to it (e.g. a publication staging tree), and leave a partial file on failure
tmp_path = OUTPUT_PATH + '.tmp'
with ProgressBar():
    output_ds.to_netcdf(tmp_path)
os.replace(tmp_path, OUTPUT_PATH)

[########################################] | 100% Completed | 17.59 s


In [8]:
output_ds

<xarray.Dataset> Size: 146MB
Dimensions:               (time: 564, bnds: 2, lat: 180, lon: 360)
Coordinates:
  * time                  (time) datetime64[ns] 5kB 1978-01-17 ... 2024-12-17
  * lat                   (lat) float64 1kB -89.5 -88.5 -87.5 ... 87.5 88.5 89.5
  * lon                   (lon) float64 3kB 0.5 1.5 2.5 ... 357.5 358.5 359.5
    forecast_period       float64 8B ...
    height                float64 8B ...
    originating_centre    |S50 50B ...
Dimensions without coordinates: bnds
Data variables:
    latitude_longitude    int32 4B ...
    time_bnds             (time, bnds) datetime64[ns] 9kB dask.array<chunksize=(564, 2), meta=np.ndarray>
    lat_bnds              (lat, bnds) float64 3kB dask.array<chunksize=(180, 2), meta=np.ndarray>
    lon_bnds              (lon, bnds) float64 6kB dask.array<chunksize=(360, 2), meta=np.ndarray>
    forecast_period_bnds  (bnds) float64 16B dask.array<chunksize=(2,), meta=np.ndarray>
    pr                    (time, lat, lon) float32 146MB dask.array<chunksize=(564, 180, 360), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.7
    title:        ERA5 monthly-mean precipitation rate, 1 degree
    source:       ERA5 monthly-mean total precipitation at 0.25 degree (ERA5/...
    history:      Conservatively regridded with xESMF 0.9.2 onto the grid of ...